#Using Python

In [0]:
from pyspark.sql import functions as F

# Create sample source/customer data
source_data = [
    (101, "John", "Mumbai", "john@gmail.com"),
    (102, "Alice", "Pune", "alice@gmail.com"),
    (103, "David", "Delhi", "david@gmail.com")
]

source_df = spark.createDataFrame(
    source_data,
    ["customer_id", "name", "city", "email"]
)

# Display source data
display(source_df)

In [0]:
# Save the source data as our initial target table
source_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("customer_target")

In [0]:
display(spark.table("customer_target"))

In [0]:
new_source_data = [
    (101, "John", "Pune", "john@gmail.com"),       # City changed
    (102, "Alice", "Pune", "alice@gmail.com"),     # No change
    (103, "David", "Delhi", "david@gmail.com"),    # No change
    (104, "Robert", "Mumbai", "robert@gmail.com")  # New customer
]

new_source_df = spark.createDataFrame(
    new_source_data,
    ["customer_id", "name", "city", "email"]
)

display(new_source_df)

In [0]:
business_key = "customer_id"

In [0]:
from delta.tables import DeltaTable

# Load the target Delta table
target_table = DeltaTable.forName(
    spark,
    "customer_target"
)

# Perform SCD Type 1 MERGE
target_table.alias("target") \
    .merge(
        new_source_df.alias("source"),
        "target.customer_id = source.customer_id"
    ) \
    .whenMatchedUpdate(set={
        "name": "source.name",
        "city": "source.city",
        "email": "source.email"
    }) \
    .whenNotMatchedInsert(values={
        "customer_id": "source.customer_id",
        "name": "source.name",
        "city": "source.city",
        "email": "source.email"
    }) \
    .execute()

In [0]:
display(
    spark.table("customer_target")
)

#Using Sql

In [0]:
%sql
CREATE OR REPLACE TABLE customer_target (
    customer_id INT,
    name STRING,
    city STRING,
    email STRING
)
USING DELTA;

In [0]:
%sql
-- Insert initial customer records
INSERT INTO customer_target VALUES
(101, 'John',  'Mumbai', 'john@gmail.com'),
(102, 'Alice', 'Pune',   'alice@gmail.com'),
(103, 'David', 'Delhi',  'david@gmail.com');

In [0]:
%sql
SELECT *
FROM customer_target;

In [0]:
%sql
CREATE OR REPLACE TABLE customer_source (
    customer_id INT,
    name STRING,
    city STRING,
    email STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO customer_source VALUES
(101, 'John',   'Pune',  'john@gmail.com'),
(102, 'Alice',  'Pune',  'alice@gmail.com'),
(103, 'David',  'Delhi', 'david@gmail.com'),
(104, 'Robert', 'Mumbai', 'robert@gmail.com');

In [0]:
%sql
MERGE INTO customer_target AS target

-- Source contains the latest records
USING customer_source AS source

-- Match records using the business key
ON target.customer_id = source.customer_id


-- =====================================================
-- WHEN MATCHED
-- =====================================================
-- Customer already exists in target.
-- Update the existing record with the latest values.
-- This is the SCD Type 1 behavior.
-- =====================================================

WHEN MATCHED THEN
UPDATE SET
    target.name  = source.name,
    target.city  = source.city,
    target.email = source.email


-- =====================================================
-- WHEN NOT MATCHED
-- =====================================================
-- Customer does not exist in target.
-- Insert it as a new record.
-- =====================================================

WHEN NOT MATCHED THEN
INSERT (
    customer_id,
    name,
    city,
    email
)
VALUES (
    source.customer_id,
    source.name,
    source.city,
    source.email
);

In [0]:
%sql
select * from customer_target;